In [ ]:
import numpy as np

from numcompute.io import load_csv
from numcompute.preprocessing import StandardScaler, SimpleImputer
from numcompute.pipeline import Pipeline
from numcompute.tree import DecisionTreeClassifier
from numcompute.ensemble import RandomForestClassifier
from numcompute.stream import StreamTrainer
from numcompute.visualise import plot_metric_over_time, compare_models

In [ ]:
# Load the dataset using the updated load_csv function
X, labels = load_csv('iris.csv')

# Encode labels to integers
unique_labels = sorted(list(set(labels)))
label_map = {lab: i for i, lab in enumerate(unique_labels)}
y = np.array([label_map[lab] for lab in labels])

print(f"dataset loaded: {len(X)} samples, {X.shape[1]} features.")
print(f"classes: {unique_labels}")

In [ ]:

# encode labels to integers
unique_labels = sorted(list(set(labels)))
label_map = {lab: i for i, lab in enumerate(unique_labels)}
y = np.array([label_map[lab] for lab in labels])

print(f"dataset loaded: {len(X)} samples, {X.shape[1]} features.")
print(f"classes: {unique_labels}")


In [ ]:

#split into chunks
chunk_size = 15
n_chunks = len(X) // chunk_size
print(f"splitting into {n_chunks} chunks of size {chunk_size}...")

# shuffling the indices
indices = np.arange(len(X))
np.random.seed(42)
np.random.shuffle(indices)
X, y = X[indices], y[indices]

X_chunks = np.array_split(X, n_chunks)
y_chunks = np.array_split(y, n_chunks)


In [ ]:

# intialise streaming and pipeline
rf_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=5, max_depth=3))
])

rf_trainer = StreamTrainer(rf_pipe)


In [ ]:

# incremental training
for i, (X_chunk, y_chunk) in enumerate(zip(X_chunks, y_chunks)):
    acc = rf_trainer.fit_chunk(X_chunk, y_chunk)
    print(f"chunk {i+1}/{n_chunks}, pre-quential Accuracy: {acc:.4f}")
    


In [ ]:
# compare with a decision tree
dt_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', StandardScaler()),
    ('model', DecisionTreeClassifier(max_depth=3))
])

dt_trainer = StreamTrainer(dt_pipe)

for i, (X_chunk, y_chunk) in enumerate(zip(X_chunks, y_chunks)):
    dt_trainer.fit_chunk(X_chunk, y_chunk)
    


In [ ]:
# visualise results
print("generating visualizations...")
rf_history = rf_trainer.get_history()['accuracy']
dt_history = dt_trainer.get_history()['accuracy']

# save plots to files
plot_metric_over_time(rf_history, title="Random Forest Accuracy Over Time", save_path="rf_accuracy.png")

compare_models(
    {'Random Forest': rf_history, 'Decision Tree': dt_history},
    save_path="model_comparison.png"
)

print("plots saved as 'rf_accuracy.png' and 'model_comparison.png'.")